In [30]:
import h5py
import torch
from torch.utils.data import Dataset, DataLoader, ConcatDataset
import utils
import torchvision.transforms as transforms
import torchmetrics
import numpy as np
from tqdm import tqdm
import torch.nn  as nn
import torch.nn.functional as F
import os
import pandas as pd
from utils.CustomDataset import PrecomputedDataset

In [2]:
TRAIN_IMAGES_FOLDER = './data/train'
VAL_IMAGES_FOLDER = './data/val'
TEST_IMAGES_FOLDER = './data_test/test'
TRAIN_IMAGES_EMBEDDINGS_FOLDER = './data/DINOv2_emb/train'
VAL_IMAGES_EMBEDDINGS_FOLDER = './data/DINOv2_emb/val'
#TEST_IMAGES_EMBEDDINGS_FOLDER = './data/DINOv2_emb/test'
SEED = 0

In [52]:
test_path = TRAIN_IMAGES_EMBEDDINGS_FOLDER + '/' + 'train_0.pth'
imgs = torch.load(test_path, weights_only=False)
img[0][0]

tensor([ 3.2400e-02,  5.7129e-01,  2.7155e-02,  1.7272e+00, -1.5953e+00,
        -4.2949e-01,  1.6972e-01, -2.4413e+00,  3.6413e+00,  1.4144e+00,
         5.3072e-01, -6.8168e-01, -5.1174e+00,  1.4181e+00, -1.3880e+00,
         4.2339e+00,  1.1390e+00,  4.4746e-01,  1.0813e-01,  2.6773e+00,
        -8.6085e-02,  4.8797e+00,  2.0598e+00, -4.3324e-01,  6.9284e-01,
         2.6191e+00,  1.6338e+00, -2.4808e-01, -8.6849e-01,  2.0975e+00,
        -1.7215e+00, -1.5097e+00, -8.4825e-01,  4.0607e+00, -1.1997e+00,
        -1.6629e+00, -1.5175e+00, -3.4453e+00, -5.0749e+00,  3.8884e+00,
         4.4935e+00,  8.8151e-01, -3.6949e+00,  6.5411e-01, -9.1239e-01,
        -8.3631e-01,  1.4767e+00,  2.8923e+00, -1.7896e+00, -3.8772e-01,
         3.5769e-01,  1.0477e+00,  1.7545e+00, -4.2722e-01, -5.0603e-01,
         3.6861e-01, -2.1706e+00,  4.4784e+00,  2.3210e+00, -1.8759e+00,
         1.8071e+00, -1.8869e+00, -2.1208e+00,  9.2484e-01, -1.9105e-01,
        -1.8972e+00,  3.3842e-01, -2.9695e+00,  2.2

In [45]:
imgs_h5 = 'train_0.h5'
file_number = imgs_h5.split('.')[0].split('_')[1]
with h5py.File(TRAIN_IMAGES_FOLDER + '/' + imgs_h5, 'r') as hdf:
    position_in_file = 0
    for img_id in list(hdf.keys()):
        print((img_id, file_number, position_in_file))

        # Extract domain information
        metadata = np.array(hdf.get(img_id).get('metadata'))
        print(int(metadata[0]))
        print(np.array(hdf.get(img_id).get('label')))
        position_in_file += 1

('0', '0', 0)
3
0
('1', '0', 1)
3
0
('10', '0', 2)
4
1
('100', '0', 3)
3
1
('1000', '0', 4)
3
0
('10000', '0', 5)
4
0
('10001', '0', 6)
4
0
('10002', '0', 7)
3
1
('10003', '0', 8)
3
0
('10004', '0', 9)
0
1
('10005', '0', 10)
0
0
('10006', '0', 11)
4
1
('10007', '0', 12)
4
1
('10008', '0', 13)
4
1
('10009', '0', 14)
3
1
('1001', '0', 15)
4
1
('10010', '0', 16)
4
1
('10011', '0', 17)
4
0
('10012', '0', 18)
0
1
('10013', '0', 19)
3
1
('10014', '0', 20)
3
1
('10015', '0', 21)
3
1
('10016', '0', 22)
0
0
('10017', '0', 23)
3
0
('10018', '0', 24)
3
1
('10019', '0', 25)
3
0
('1002', '0', 26)
0
0
('10020', '0', 27)
4
1
('10021', '0', 28)
4
0
('10022', '0', 29)
0
1
('10023', '0', 30)
4
1
('10024', '0', 31)
4
0
('10025', '0', 32)
0
0
('10026', '0', 33)
4
1
('10027', '0', 34)
4
1
('10028', '0', 35)
4
1
('10029', '0', 36)
0
1
('1003', '0', 37)
0
0
('10030', '0', 38)
4
0
('10031', '0', 39)
3
0
('10032', '0', 40)
4
0
('10033', '0', 41)
0
1
('10034', '0', 42)
3
1
('10035', '0', 43)
3
1
('10036', '0', 

# Compute embeddings

In [3]:
def precompute(dataloader, model, device):
    xs, ys = [], []
    for x, y in tqdm(dataloader, leave=False):
        with torch.no_grad():
            xs.append(model(x.to(device)).detach().cpu().numpy())
        ys.append(y.numpy())
    xs = np.vstack(xs)
    ys = np.hstack(ys)
    return torch.tensor(xs), torch.tensor(ys)

In [82]:
from importlib import reload
reload(utils.CustomDataset)

<module 'utils.CustomDataset' from 'c:\\Users\\tlc29\\Documents\\CS\\3A\\Medical Imaging\\MVA-DLMI-2025---Histopathology-OOD-classification\\utils\\CustomDataset.py'>

In [83]:
from utils.CustomDataset import EmbeddedDomainAwareDataset

In [ ]:
# # Initialize datasets and dataloaders
# BATCH_SIZE = 16
# print("Make training dataset :")
# train_dataset = EmbeddedDomainAwareDataset(TRAIN_IMAGES_FOLDER, TRAIN_IMAGES_EMBEDDINGS_FOLDER, None, 'train')
# print("Make validation dataset :")
# val_dataset = EmbeddedDomainAwareDataset(VAL_IMAGES_FOLDER, VAL_IMAGES_EMBEDDINGS_FOLDER, None, 'val')

# train_dataloader = DataLoader(train_dataset, shuffle=True, batch_size=BATCH_SIZE)
# val_dataloader = DataLoader(val_dataset, shuffle=False, batch_size=BATCH_SIZE)

Make training dataset :
Make validation dataset :


In [ ]:
# torch.save(train_dataset, 'embedded_center_aware_dataset.pth')
# torch.save(val_dataset, 'embedded_center_aware_dataset_val.pth')

In [ ]:
train_dataset = torch.load(TRAIN_IMAGES_PATH, weights_only=False)
val_dataset = torch.load(VAL_IMAGES_PATH, weights_only=False)

In [59]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Working on {device}.')

Working on cpu.


In [60]:
num_domains = len(np.unique(train_dataset.domains))

In [61]:
import torch

class GradientReversalLayer(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, alpha):
        ctx.alpha = alpha
        return x.view_as(x) # Avoid modifying the input tensor directly

    @staticmethod
    def backward(ctx, grad_output):
        # Return the negative gradient multiplied by alpha, and None for alpha's gradient
        return grad_output.neg() * ctx.alpha, None


In [97]:

class DomainInvariantModel(torch.nn.Module):
    def __init__(self, feature_dim, num_domains):
        super(DomainInvariantModel, self).__init__()
   
     # Task classifier
        self.task_classifier = torch.nn.Sequential(
            torch.nn.Linear(feature_dim, 1),
            torch.nn.Sigmoid()
        )
        
        # Domain classifier
        self.domain_classifier = torch.nn.Sequential(
            torch.nn.Linear(feature_dim, 512),
            torch.nn.ReLU(),
            torch.nn.Linear(512, num_domains),
            torch.nn.Softmax(dim=1)  # Softmax for domain classification to output probabilities
        )
        
        # Gradient reversal layer
        self.grl = GradientReversalLayer
    
    def forward(self, x, alpha=1.0):
        features = x
        # Task prediction
        task_output = self.task_classifier(features)
        
        # Domain prediction (with gradient reversal)
        reversed_features = self.grl.apply(features, alpha)
        domain_output = self.domain_classifier(reversed_features)
        
        return task_output, domain_output


In [98]:
model = DomainInvariantModel(len(train_dataset[0][0]), num_domains).to(device)

In [99]:
OPTIMIZER = 'Adam'
OPTIMIZER_PARAMS = {'lr': 0.001}
TASK_LOSS = 'BCELoss'
DOMAIN_LOSS = 'CrossEntropyLoss'
METRIC = 'Accuracy'
NUM_EPOCHS = 100
PATIENCE = 10
LAMBDA = 0.1  # Weight for domain adaptation loss


optimizer = getattr(torch.optim, OPTIMIZER)(model.parameters(), **OPTIMIZER_PARAMS)
task_criterion = getattr(torch.nn, TASK_LOSS)()
domain_criterion = getattr(torch.nn, DOMAIN_LOSS)()
metric = getattr(torchmetrics, METRIC)('binary')
min_loss, best_epoch = float('inf'), 0

In [96]:

for epoch in range(NUM_EPOCHS):
    model.train()
    train_metrics, train_losses = [], []
    
    # Gradually increase the domain adaptation weight
    alpha = 2. * (epoch / NUM_EPOCHS) - 1 if epoch <= NUM_EPOCHS / 2 else 1.0
    
    for train_x, train_y, train_d in tqdm(train_dataloader, leave=False):
        optimizer.zero_grad()
        
        # Forward pass
        task_pred, domain_pred = model(train_x.to(device), alpha)
        
        # Calculate losses
        task_loss = task_criterion(task_pred, train_y.to(device).view(-1, 1).float())
        # Domain loss (cross-entropy)
        train_d = F.one_hot(train_d // 2, num_classes=num_domains).float().to(device).view(-1, num_domains)
        domain_loss = domain_criterion(domain_pred, train_d.to(device))
        total_loss = task_loss + LAMBDA * domain_loss
        
        # Backward pass
        total_loss.backward()
        optimizer.step()
        
        # Track metrics
        train_losses.extend([task_loss.item()] * len(train_y))
        train_metric = metric(task_pred.cpu(), train_y.int().cpu())
        train_metrics.extend([train_metric.item()] * len(train_y))
    
    print(f'Epoch train [{epoch+1}/{NUM_EPOCHS}] | Loss {np.mean(train_losses):.4f} | Metric {np.mean(train_metrics):.4f}')


    # Validation (without domain adaptation)
    model.eval()
    val_metrics, val_losses = [], []
    for val_x, val_y, _ in tqdm(val_dataloader, leave=False):
        with torch.no_grad():
            val_pred, _ = model(val_x.to(device), 0)  # alpha=0 disables domain adaptation
            
        loss = task_criterion(val_pred, val_y.to(device))
        val_losses.extend([loss.item()] * len(val_y))
        val_metric = metric(val_pred.cpu(), val_y.int().cpu())
        val_metrics.extend([val_metric.item()] * len(val_y))
    
    print(f'Epoch valid [{epoch+1}/{NUM_EPOCHS}] | Loss {np.mean(val_losses):.4f} | Metric {np.mean(val_metrics):.4f}')

    if np.mean(val_losses) < min_loss:
        mean_val_loss = np.mean(val_losses)
        print(f'New best loss {min_loss:.4f} -> {mean_val_loss:.4f}')
        min_loss = mean_val_loss
        best_epoch = epoch
        torch.save(model.state_dict(), 'best_model.pth')

    if epoch - best_epoch == PATIENCE:
        break



Epoch train [1/100] | Loss 0.1821 | Metric 0.9307


Epoch valid [1/100] | Loss 0.3680 | Metric 0.8499
New best loss inf -> 0.3680


Epoch train [2/100] | Loss 0.1568 | Metric 0.9412


Epoch valid [2/100] | Loss 0.3698 | Metric 0.8517


KeyboardInterrupt: 

In [ ]:
model.load_state_dict(torch.load('best_model.pth', weights_only=True))
model.eval()
model.to(device)
with h5py.File(TEST_IMAGES_PATH, 'r') as hdf:
    test_ids = list(hdf.keys())

solutions_data = {'ID': [], 'Pred': []}
with h5py.File(TEST_IMAGES_PATH, 'r') as hdf:
    for test_id in tqdm(test_ids):
        img = preprocessing(torch.tensor(np.array(hdf.get(test_id).get('img')))).unsqueeze(0).float()
        with torch.no_grad():
            features = feature_extractor(img.to(device))
            pred, _ = model(features, 0)  # alpha=0 for inference
        solutions_data['ID'].append(int(test_id))
        solutions_data['Pred'].append(int(pred.item() > 0.5))
solutions_data = pd.DataFrame(solutions_data).set_index('ID')
solutions_data.to_csv('domain_invariant_adaptation.csv')

# Final predictions

To create a solutions file, you need to generate a CSV with 2 columns.
- **ID**: containing the ID of the image
- **Pred**: with the predicted class (**threshold the prediction to get either 0 or 1**)

In [98]:
torch.load("data/DINOv2_emb/test/test_0.pth").features.shape

C:\Users\Chardin Pierre\AppData\Local\Temp\ipykernel_12012\1148655247.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  torch.load("data/DINOv2_emb/test/test_0.pth").featu

torch.Size([86, 384])

In [47]:
linear_probing.load_state_dict(torch.load('DINO_baseline/best_model.pth', weights_only=True))
linear_probing.eval()
linear_probing.to(device)
prediction_dict = {}

In [112]:
def predict(model, path):
    solutions_data = {'ID': [], 'Pred': []}
    preprocessing = transforms.Resize((98, 98))

    with h5py.File(path, 'r') as hdf:
        for test_id in list(hdf.keys()):
            img = preprocessing(torch.tensor(np.array(hdf.get(test_id).get('img')))).unsqueeze(0).float()
            pred = model(feature_extractor(img.to(device))).detach().cpu()
            solutions_data['ID'].append(int(test_id))
            solutions_data['Pred'].append(int(pred.item() > 0.5))
    return solutions_data['ID'], solutions_data['Pred']

In [113]:
predictions = [predict(linear_probing, f"data/test/{file}") for file in tqdm(os.listdir("data/test"))]

100%|██████████| 1000/1000 [5:44:47<00:00, 20.69s/it] 


In [114]:
solutions_data = pd.DataFrame({'ID': np.concatenate([np.array(i) for i,_ in predictions]),
                                'Pred': np.concatenate([np.array(p) for _,p in predictions])})\
                        .set_index('ID')\
                        .sort_index()

In [115]:
solutions_data.to_csv('DINO_baseline/baseline.csv')

In [116]:
solutions_data

,Pred
ID,
0,0
1,0
2,0
3,1
4,0
...,...
85049,0
85050,1
85051,0
